# PEFT 前的准备
首先是安装的版本：
- transformers: 4.57.1
- peft: 0.17.1
- datasets: 4.2.0

In [1]:
import transformers, peft, datasets
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("datasets:", datasets.__version__)

transformers: 4.57.1
peft: 0.17.1
datasets: 4.2.0


In [2]:
!pwd

/root/autodl-tmp


In [25]:
from datasets import Dataset, load_dataset
data_path = "/root/autodl-tmp/dataset/alpaca-gpt4-data-zh/alpaca_gpt4_data_zh.json"
ds = load_dataset("json", data_files=data_path, split="train")

In [27]:
ds, ds[0]

(Dataset({
     features: ['instruction', 'input', 'output'],
     num_rows: 48818
 }),
 {'instruction': '保持健康的三个提示。',
  'input': '',
  'output': '以下是保持健康的三个提示：\n\n1. 保持身体活动。每天做适当的身体运动，如散步、跑步或游泳，能促进心血管健康，增强肌肉力量，并有助于减少体重。\n\n2. 均衡饮食。每天食用新鲜的蔬菜、水果、全谷物和脂肪含量低的蛋白质食物，避免高糖、高脂肪和加工食品，以保持健康的饮食习惯。\n\n3. 睡眠充足。睡眠对人体健康至关重要，成年人每天应保证 7-8 小时的睡眠。良好的睡眠有助于减轻压力，促进身体恢复，并提高注意力和记忆力。'})

In [31]:
from transformers import AutoModelForCausalLM, AutoTokenizer

path = "model/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(path)

In [33]:
tokenizer.special_tokens_map, tokenizer.eos_token

({'eos_token': '<|im_end|>',
  'pad_token': '<|endoftext|>',
  'additional_special_tokens': ['<|im_start|>',
   '<|im_end|>',
   '<|object_ref_start|>',
   '<|object_ref_end|>',
   '<|box_start|>',
   '<|box_end|>',
   '<|quad_start|>',
   '<|quad_end|>',
   '<|vision_start|>',
   '<|vision_end|>',
   '<|vision_pad|>',
   '<|image_pad|>',
   '<|video_pad|>']},
 '<|im_end|>')

In [34]:
def process_func(example):
    MAX_LENGTH = 256
    input_ids, attention_mask, labels = [], [], []

    instruction = tokenizer(
        "\n".join(["Human: " + example["instruction"], example["input"]]).strip() + "\n\nAssistant: "
    )

    response = tokenizer(example["output"] + tokenizer.eos_token)
    input_ids = instruction["input_ids"] + response["input_ids"]
    attention_mask = instruction["attention_mask"] + response["attention_mask"]
    labels = [-100] * len(instruction["input_ids"]) + response["input_ids"]

    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
        
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [35]:
tokenized_ds = ds.map(process_func, remove_columns=ds.column_names)
tokenized_ds

Map:   0%|          | 0/48818 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 48818
})

In [36]:
tokenizer.decode(tokenized_ds[1]["input_ids"])

'Human: 三原色是什么？\n\nAssistant: 三原色通常指的是红色、绿色和蓝色（RGB）。它们是通过加色混合原理创建色彩的三种基础颜色。在以发光为基础的显示设备中（如电视、计算机显示器、智能手机和平板电脑显示屏）, 三原色可混合产生大量色彩。其中红色和绿色可以混合生成黄色，红色和蓝色可以混合生成品红色，蓝色和绿色可以混合生成青色。当红色、绿色和蓝色按相等比例混合时，可以产生白色或灰色。\n\n此外，在印刷和绘画中，三原色指的是以颜料为基础的红、黄和蓝颜色（RYB）。这三种颜色用以通过减色混合原理来创建色彩。不过，三原色的具体定义并不唯一，不同的颜色系统可能会采用不同的三原色。<|im_end|>'

In [37]:
tokenizer.decode(list(filter(lambda x: x != -100, tokenized_ds[1]["labels"])))

'三原色通常指的是红色、绿色和蓝色（RGB）。它们是通过加色混合原理创建色彩的三种基础颜色。在以发光为基础的显示设备中（如电视、计算机显示器、智能手机和平板电脑显示屏）, 三原色可混合产生大量色彩。其中红色和绿色可以混合生成黄色，红色和蓝色可以混合生成品红色，蓝色和绿色可以混合生成青色。当红色、绿色和蓝色按相等比例混合时，可以产生白色或灰色。\n\n此外，在印刷和绘画中，三原色指的是以颜料为基础的红、黄和蓝颜色（RYB）。这三种颜色用以通过减色混合原理来创建色彩。不过，三原色的具体定义并不唯一，不同的颜色系统可能会采用不同的三原色。<|im_end|>'